# Hi! 👋

The following notebook is used to **visualize experiments' latent spaces**. If you are running the following notebook in Colab, the next cell will ask you for a GDrive URL and a github token. The token will be used to clone [DecentralizedLearning/dEngine](https://github.com/DecentralizedLearning/dEngine). If you don't own the repo, the token can be generated using Github web (`settings > Developer Settings > Personal Access Tokens`). The GDrive URL will be provided by who ran the experiments. 

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import gdown

try:  # Check if we are in Google colab
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
    
    !git clone --quiet https://github.com/DecentralizedLearning/dEngine ..
    !pip install --quiet git+https://github.com/DecentralizedLearning/notebooks.git
    
    EXPERIMENT_GDRIVE_URL = input('Experiment remote URL \t')
    LOGS_PATH = Path('/content/logs')
    if not os.path.exists(LOGS_PATH):
        gdown.download_folder(url=EXPERIMENT_GDRIVE_URL, output=str(LOGS_PATH))
        !unzip -q {LOGS_PATH}/\*.zip -d {LOGS_PATH}    
except:
    pass

<br><br>
<hr>

<br><br>

# Experiment Selection

The next cell will open a file selection widget.

Please select the directory containing the experiment logs (usually inside a `log/` folder).

**Note**: If you are using Google Drive, you may need to navigate one level up. Logs are typically downloaded to `/content/logs`, while the code runs in `/content/SaiSim`.

In [ ]:
from dnotebooks.widgets import MultiExperimentSelection

experiment_selection_widget, get_selected_experiments = MultiExperimentSelection(limit=1)
display(experiment_selection_widget)

<br><br>
<hr>
<br><br>

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = [12, 6]  # wide figure
plt.rcParams['figure.dpi'] = 100
plt.close('all') 

from dengine.analysis import ExperimentConfusionMatrix
from dengine.analysis.experiment import Experiment
from dnotebooks.widgets import LayerActivationDashboard

selected_experiment = Experiment(
   experiment_root_path=get_selected_experiments()[0], 
   dataset_root_path=Path("./datasets/")
)

selected_experiment_root_path = selected_experiment.experiment_cfg.output_directory
dashboard = LayerActivationDashboard(experiment=selected_experiment)
display(dashboard.render())

<br><br>
<hr>
<br><br>

### UMAP, projection of all targets

In [ ]:
%matplotlib widget

plt.close('all')
plt.rcParams['figure.figsize'] = [8, 8]  # wide figure
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.format'] = 'svg'

from umap import UMAP
from torch.utils.data import Subset

from dengine.analysis.experiment import LegacyExperiment

from dnotebooks.widgets import UmapWidget


# ............... ............... ............... #
# COMPUTE EMBEDDINGS
# ............... ............... ............... #
selected_node = str(dashboard.interactive_graph.selection[0])
model = selected_experiment.training_engine.clients[selected_node].model
D = selected_experiment.dataset_test

activations = dashboard.get_activations(D).squeeze()
embedding = UMAP(random_state=42).fit_transform(activations)

In [ ]:
title = f'UMAP Node-{selected_node} layer-{dashboard._layer_selection_dropdown.value} ({selected_experiment_root_path.name})'
cifar10_labels = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}
highlight_labels = []
plot = UmapWidget(D, embedding, labels=cifar10_labels, title="My cool experiment title - Test Set", highlight_labels=highlight_labels)
plot.render()

<br><br>
<hr>

<br><br>